<a href="https://colab.research.google.com/github/zwimpee/cursivetransformer/blob/main/cursivetransformer_mech_interp_part_3_stroke_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualizing Attention on Stroke Tokens

# Setup

In [3]:
!pip install transformer_lens
!pip install gradio
!pip install wandb
!pip install einops
!pip install matplotlib
!pip install datasets

# Clone the cursivetransformer repository and install its requirements
!rm -rf cursivetransformer && git clone https://github.com/zwimpee/cursivetransformer.git
!pip install -r cursivetransformer/requirements.txt

Cloning into 'cursivetransformer'...
remote: Enumerating objects: 2926, done.
remote: Counting objects: 100% (447/447), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 2926 (delta 423), reused 385 (delta 378), pack-reused 2479 (from 1)
Receiving objects: 100% (2926/2926), 64.06 MiB | 15.60 MiB/s, done.
Resolving deltas: 100% (1620/1620), done.
  Cloning https://github.com/callummcdougall/CircuitsVis.git to /tmp/pip-req-build-pdqp0nku
  Running command git clone --filter=blob:none --quiet https://github.com/callummcdougall/CircuitsVis.git /tmp/pip-req-build-pdqp0nku
  Resolved https://github.com/callummcdougall/CircuitsVis.git to commit 1e6129d08cae7af9242d9ab5d3ed322dd44b4dd3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import sys
sys.path.append('/content/cursivetransformer')
from cursivetransformer.model import get_all_args, get_checkpoint, get_latest_checkpoint_artifact
from cursivetransformer.data import create_datasets, offsets_to_strokes, strokes_to_offsets
from cursivetransformer.sample import generate, generate_n_words, plot_strokes
from cursivetransformer.mech_interp import (
    HookedCursiveTransformer,
    HookedCursiveTransformerConfig,
    convert_cursivetransformer_model_config,
    visualize_attention,
    generate_repeated_stroke_tokens,
    generate_random_ascii_context,
    run_and_cache_model_repeated_tokens,
    compute_induction_scores,
    plot_induction_scores,
    plot_head_attention_pattern,
    create_induction_summary,
    ablate_heads,
    get_induction_positions,
    compute_loss_on_induction_positions
)

import pandas as pd
import os

import copy
import types
from typing import List, Callable, Dict, Optional, Union, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import einops
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import circuitsvis as cv
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from jaxtyping import Float, Int


import transformer_lens.utils as utils
from transformer_lens.hook_points import HookPoint
from transformer_lens import ActivationCache

torch.set_grad_enabled(False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import wandb
wandb.login()

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [6]:
args = get_all_args(False)
args.sample_only = True
args.max_seq_length = 1250
args.load_from_run_id = '7e9hz1og'
args.wandb_project = 'bigbank_2k'
args.wandb_entity = 'sam-greydanus'
args.dataset_name = 'bigbank'
args.wandb_run_name = 'cursivetransformer_mech_interp_part_3_stroke_attention'
args.wandb_api_key = 'e56bbe426145e5983e72a933299daca195ebb6a7'
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
train_dataset, test_dataset = create_datasets(args)
args.vocab_size = train_dataset.get_vocab_size()
args.block_size = train_dataset.get_stroke_seq_length()
args.context_block_size = train_dataset.get_text_seq_length()
args.context_vocab_size = train_dataset.get_char_vocab_size()

Enter your W&B API key: ··········
Trying to load dataset file from /content/cursivetransformer/data/bigbank.json.zip
Succeeded in loading the bigbank dataset; contains 2000 items.
For a dataset of 1900 examples we can generate 205257574037880 combinations of 5 examples.
Generating 497000 random combinations.
For a dataset of 100 examples we can generate 75287520 combinations of 5 examples.
Generating 3000 random combinations.
Number of examples in the train dataset: 497000
Number of examples in the test dataset: 3000
Max token sequence length: 1250
Number of unique characters in the ascii vocabulary: 71
Ascii vocabulary:
	" enaitoshrdx.vpukbgfcymzw1lqj804I92637OTAS5N)EHR"'(BCQLMWYU,ZF!DXV?KPGJ"
Split up the dataset into 497000 training examples and 3000 test examples


In [9]:
text = "Hello, Sam!"

# Load original model and plot generated strokes

In [ ]:
# - [x] TODO: Load original model and plot generated strokes
model, optimizer, scheduler, step, best_loss = get_checkpoint(args, sample_only=True)

In [ ]:
offset_samp, point_samp, _ = generate_n_words(model, test_dataset, text, n_words=2)

In [ ]:
_ = plot_strokes(point_samp, text)

## Load model into HookedCursiveTransformer

In [7]:
cfg = convert_cursivetransformer_model_config(args)
model = HookedCursiveTransformer.from_pretrained("cursivetransformer", cfg)

Loading pretrained model cursivetransformer
Finding latest checkpoint for W&B run id 7e9hz1og
  model:best_checkpoint:v129
  model:best_checkpoint:v130
  model:best_checkpoint:v131
  model:best_checkpoint:v132
  model:best_checkpoint:v133
  model:best_checkpoint:v134
  model:best_checkpoint:v135
  model:best_checkpoint:v136
  model:best_checkpoint:v137
  model:best_checkpoint:v138
  model:best_checkpoint:v139
  model:best_checkpoint:v140
  model:best_checkpoint:v141
  model:best_checkpoint:v142
  model:best_checkpoint:v143
  model:best_checkpoint:v144
  model:best_checkpoint:v145
  model:best_checkpoint:v146
  model:best_checkpoint:v147
  model:best_checkpoint:v148
  wandb-history:run-7e9hz1og-history:v4
Selected:  model:best_checkpoint:v148


wandb:   1 of 1 files downloaded.  


Successfully loaded pretrained model cursivetransformer


# Generate strokes using the hooked model

In [10]:
offset_samp, point_samp, cache = generate_n_words(model, test_dataset, text, n_words=2)

cpu


AttributeError: 'HookedCursiveTransformerConfig' object has no attribute 'block_size'

# Extract attention patterns

In [ ]:
attn_patterns = torch.stack([cache[f'blocks.{i}.attn.hook_pattern'] for i in range(model.cfg.n_layer)])
attn_patterns = attn_patterns.mean(dim=1)  # Average across heads

# Create a colormap

In [ ]:
cmap = plt.get_cmap('viridis')

# Plot strokes with attention-based coloring

In [ ]:
fig, ax = plt.subplots(figsize=(18, 3))

for i in range(len(point_samp) - 1):
    x1, y1, _ = point_samp[i]
    x2, y2, _ = point_samp[i+1]

    # Get attention pattern for predicting the next token
    attn = attn_patterns[:, i+1, :i+1].mean(dim=0)

    # Normalize attention to [0, 1] range for coloring
    attn_normalized = (attn - attn.min()) / (attn.max() - attn.min())

    # Create a gradient of colors based on attention
    colors = [cmap(a.item()) for a in attn_normalized]

    # Plot the stroke with a gradient of colors
    points = np.array([[x1, 1-y1], [x2, 1-y2]])
    segments = np.concatenate([points[:-1, np.newaxis, :], points[1:, np.newaxis, :]], axis=1)
    lc = LineCollection(segments, colors=colors, linewidth=1.3)
    ax.add_collection(lc)

ax.set_xlim(min(p[0] for p in point_samp), max(p[0] for p in point_samp))
ax.set_ylim(min(1-p[1] for p in point_samp), max(1-p[1] for p in point_samp))
ax.set_aspect('equal')
ax.set_title(f'Strokes for "{text}" with attention-based coloring')

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('Attention Strength')

plt.show()

# Plot attention heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(attn_patterns.mean(dim=0).cpu(), cmap='viridis')
plt.title('Attention Heatmap')
plt.xlabel('Source Token')
plt.ylabel('Target Token')
plt.show()